In [31]:
import pandas as pd

In [32]:
df = pd.read_csv('../data/row_data/itsm.csv')

C:\Users\rashi\AppData\Local\Temp\ipykernel_19232\1476173655.py:1: DtypeWarning: Columns (0: Urgency) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/row_data/itsm.csv')


In [33]:
df.isnull().sum()

CI_Name                           0
CI_Cat                          111
CI_Subcat                       111
WBS                               0
Incident_ID                       0
Status                            0
Impact                            0
Urgency                           0
Priority                       1380
number_cnt                        0
Category                          0
KB_number                         0
Alert_Status                      0
No_of_Reassignments               1
Open_Time                         0
Reopen_Time                   44322
Resolved_Time                  1780
Close_Time                        0
Handle_Time_hrs                   1
Closure_Code                    460
No_of_Related_Interactions      114
Related_Interaction               0
No_of_Related_Incidents       45384
No_of_Related_Changes         46046
Related_Change                46046
dtype: int64

## Drop columns

In [34]:
df.columns

Index(['CI_Name', 'CI_Cat', 'CI_Subcat', 'WBS', 'Incident_ID', 'Status',
       'Impact', 'Urgency', 'Priority', 'number_cnt', 'Category', 'KB_number',
       'Alert_Status', 'No_of_Reassignments', 'Open_Time', 'Reopen_Time',
       'Resolved_Time', 'Close_Time', 'Handle_Time_hrs', 'Closure_Code',
       'No_of_Related_Interactions', 'Related_Interaction',
       'No_of_Related_Incidents', 'No_of_Related_Changes', 'Related_Change'],
      dtype='str')

In [35]:
df = df.drop(columns=['Reopen_Time','No_of_Related_Incidents','No_of_Related_Changes','Related_Change'])

In [36]:
df.columns

Index(['CI_Name', 'CI_Cat', 'CI_Subcat', 'WBS', 'Incident_ID', 'Status',
       'Impact', 'Urgency', 'Priority', 'number_cnt', 'Category', 'KB_number',
       'Alert_Status', 'No_of_Reassignments', 'Open_Time', 'Resolved_Time',
       'Close_Time', 'Handle_Time_hrs', 'Closure_Code',
       'No_of_Related_Interactions', 'Related_Interaction'],
      dtype='str')

## drop missing values

In [37]:
df = df.dropna(subset=['CI_Cat','CI_Subcat','No_of_Reassignments', 'Handle_Time_hrs','Priority'])

## Fill null value

In [38]:
df['Closure_Code'] = df['Closure_Code'].fillna('Unknown')

df['No_of_Related_Interactions'] = df['No_of_Related_Interactions'].fillna(df['No_of_Related_Interactions'].median())

In [39]:
df.isnull().sum()

CI_Name                          0
CI_Cat                           0
CI_Subcat                        0
WBS                              0
Incident_ID                      0
Status                           0
Impact                           0
Urgency                          0
Priority                         0
number_cnt                       0
Category                         0
KB_number                        0
Alert_Status                     0
No_of_Reassignments              0
Open_Time                        0
Resolved_Time                 1634
Close_Time                       0
Handle_Time_hrs                  0
Closure_Code                     0
No_of_Related_Interactions       0
Related_Interaction              0
dtype: int64

In [40]:
df['Urgency'].unique()

array([4, 3, 5, 2, 1, '5', '3', '4', '2', '1', '5 - Very Low'],
      dtype=object)

In [41]:
df['Impact'].unique()

<StringArray>
['4', '3', '5', '2', '1']
Length: 5, dtype: str

## convert the tex leble into number

In [42]:
df['Urgency'] = df['Urgency'].astype(str).str.strip()
df['Urgency'] = df['Urgency'].str.extract(r'(\d+)')
df['Urgency'] = df['Urgency'].astype(int)

print(df['Urgency'].unique())
print(df['Urgency'].value_counts())

[4 3 5 2 1]
Urgency
4    22529
5    16719
3     5175
2      688
1        6
Name: count, dtype: int64


In [43]:
df['Impact'] = df['Impact'].astype(str).str.strip()
df['Impact'] = df['Impact'].str.extract(r'(\d+)')
df['Impact'] = df['Impact'].astype(int)

print(df['Impact'].unique())

[4 3 5 2 1]


In [44]:
df['Handle_Time_hrs'].head()

0    3,87,16,91,111
1    4,35,47,86,389
3    4,32,18,33,333
4    3,38,39,03,333
5    3,38,34,36,944
Name: Handle_Time_hrs, dtype: str

In [45]:
df['Open_Time'] = pd.to_datetime(df['Open_Time'], format='%d-%m-%Y %H:%M', errors='coerce')
df['Close_Time'] = pd.to_datetime(df['Close_Time'], format='%d-%m-%Y %H:%M', errors='coerce')

df['Handle_Time_hrs'] = (df['Close_Time'] - df['Open_Time']).dt.total_seconds() / 3600

print(df['Handle_Time_hrs'].describe())
print(df['Handle_Time_hrs'].isnull().sum())
print((df['Handle_Time_hrs'] < 0).sum())

count    45117.000000
mean       124.141921
std        448.203876
min          0.000000
25%          1.316667
50%         18.800000
75%         95.800000
max      15312.316667
Name: Handle_Time_hrs, dtype: float64
0
0


## Data Cleaning — Observations

### Initial Data Quality Check
On inspecting the extracted dataset (46,606 rows, 25 columns), several data quality issues were identified across multiple columns, requiring cleaning before proceeding to analysis.

### Missing Values Summary

| Column | Missing Count | % Missing | Action Taken |
|---|---|---|---|
| `Reopen_Time` | 44,322 | 95.1% | Dropped column |
| `No_of_Related_Incidents` | 45,384 | 97.4% | Dropped column |
| `No_of_Related_Changes` | 46,046 | 98.8% | Dropped column |
| `Related_Change` | 46,046 | 98.8% | Dropped column |
| `CI_Cat`, `CI_Subcat` | 111 each | 0.24% | Rows dropped (negligible impact) |
| `Priority` (target variable) | 1,380 | 2.96% | Rows dropped — cannot impute the target label |
| `No_of_Reassignments`, `Handle_Time_hrs` | 1 each | ~0% | Rows dropped |
| `Closure_Code` | 460 | 0.99% | Filled with `"Unknown"` |
| `No_of_Related_Interactions` | 114 | 0.24% | Filled with median value |
| `Resolved_Time` | 1,780 | 3.8% | Left as-is — likely represents tickets not yet resolved; excluded from modeling to avoid data leakage |

### Data Inconsistency: `Urgency` Column
The `Urgency` column was found to contain mixed data types — integers (`4, 3, 5`), string numbers (`'4', '3'`), and descriptive text labels (`'5 - Very Low'`) — all representing the same underlying scale (1–5). This caused pandas to infer the column as an `object` dtype instead of a clean numeric type.

**Fix:** Extracted the leading numeric value from each entry using regex and converted the column to a consistent integer type. The `Impact` column was also checked for the same issue.

### Data Corruption: `Handle_Time_hrs` Column
The `Handle_Time_hrs` column contained severely malformed values (e.g., `"3,87,16,91,111"` instead of an expected decimal value like `3871.69`). The comma-based decimal notation (used in the source system) appears to have been affected by locale-based number formatting at the database/export level, resulting in Indian-style thousands-grouping being applied to what should have been a single decimal number. This corruption was found to originate from the source data itself, not from post-export handling.

**Fix:** Rather than attempting to repair the corrupted text values, `Handle_Time_hrs` was **recalculated from scratch** using the difference between `Close_Time` and `Open_Time` (converted to proper `datetime` format), expressed in hours. This approach was more reliable and avoided dependency on inconsistent source formatting.

In [46]:
df.head()

,CI_Name,CI_Cat,CI_Subcat,WBS,Incident_ID,Status,Impact,Urgency,Priority,number_cnt,...,KB_number,Alert_Status,No_of_Reassignments,Open_Time,Resolved_Time,Close_Time,Handle_Time_hrs,Closure_Code,No_of_Related_Interactions,Related_Interaction
0,SUB000508,subapplication,Web Based Application,WBS000162,IM0000004,Closed,4,4,4.0,0.601292,...,KM0000553,closed,26.0,2012-02-05 13:32:00,04-11-2013 13:50,2013-11-04 13:51:00,15312.316667,Other,1.0,SD0000007
1,WBA000124,application,Web Based Application,WBS000088,IM0000005,Closed,3,3,3.0,0.415050,...,KM0000611,closed,33.0,2012-03-12 15:44:00,02-12-2013 12:36,2013-12-02 12:36:00,15116.866667,Software,1.0,SD0000011
3,WBA000124,application,Web Based Application,WBS000088,IM0000011,Closed,4,4,4.0,0.642927,...,KM0000611,closed,13.0,2012-07-17 11:49:00,14-11-2013 09:31,2013-11-14 09:31:00,11637.700000,Operator error,1.0,SD0000025
4,WBA000124,application,Web Based Application,WBS000088,IM0000012,Closed,4,4,4.0,0.345258,...,KM0000611,closed,2.0,2012-08-10 11:01:00,08-11-2013 13:55,2013-11-08 13:55:00,10922.900000,Other,1.0,SD0000029
5,WBA000124,application,Web Based Application,WBS000088,IM0000013,Closed,4,4,4.0,0.006676,...,KM0000611,closed,4.0,2012-08-10 11:27:00,08-11-2013 13:54,2013-11-08 13:54:00,10922.450000,Other,1.0,SD0000031


In [47]:
df['Resolved_Time'] = pd.to_datetime(df['Resolved_Time'], format='%d-%m-%Y %H:%M', errors='coerce')
df['Resolved_Time'].head()

0   2013-11-04 13:50:00
1   2013-12-02 12:36:00
3   2013-11-14 09:31:00
4   2013-11-08 13:55:00
5   2013-11-08 13:54:00
Name: Resolved_Time, dtype: datetime64[us]

In [48]:
df['Resolved_Time'].isnull().sum()

np.int64(1634)

In [49]:
print(df['Priority'].unique())
print(df['Urgency'].unique())
print(df['Impact'].unique())

[4. 3. 5. 2. 1.]
[4 3 5 2 1]
[4 3 5 2 1]


In [50]:
df.duplicated().sum()

np.int64(0)

All three columns now contain exactly 5 valid, consistent values (1–5), matching the Priority Matrix defined in the project documentation.

---

### 3. Datetime Standardization

`Open_Time`, `Resolved_Time`, and `Close_Time` were originally stored as text strings in inconsistent formats. All three were converted to proper `datetime` objects using `pd.to_datetime()`. Missing values in `Resolved_Time` were preserved as `NaT` (no data loss or distortion introduced).


**Post-recalculation summary:**

| Statistic | Value |
|---|---|
| Count | 45,117 |
| Mean | 124.14 hrs |
| Median | 18.80 hrs |
| Min | 0 hrs |
| Max | 15,312.32 hrs |
| Negative values | 0 |

The distribution is right-skewed, consistent with real-world incident handling patterns — most tickets close within a day, while a small number of long-running tickets pull the average upward. No negative durations were found, confirming logical consistency between timestamps.

---

### Final Verification
- Rechecked `Priority`, `Urgency`, and `Impact` for valid range (1–5) — confirmed clean.
- Checked for duplicate records using `df.duplicated().sum()` — no duplicates found requiring removal.
- Final cleaned dataset shape confirmed and saved to `data/processed/itsm_clean.csv`.

---

### Conclusion
The dataset has been thoroughly cleaned and standardized. Key categorical and numeric fields (`Priority`, `Impact`, `Urgency`) are now consistent and analysis-ready. The `Handle_Time_hrs` feature was reconstructed reliably from verified timestamp data. The cleaned dataset is now ready for **Exploratory Data Analysis (EDA)**.

In [51]:
df.to_csv('../data/process_data/itsm_clean.csv', index=False)